# 02_local_inference_optimization

Transitioning from cloud-managed APIs to local model inference (using runtimes like Ollama or LM Studio) is a core competency expected of modern GenAI engineers. Companies look for developers who understand data privacy compliance (HIPAA, GDPR), cost governance, and physical hardware constraints.

## 1. The VRAM Sizing & Quantization Math
Interviewers love asking: "If we want to run a 70B parameter model locally, how much VRAM do we need, and how does quantization alter that math?"

**Raw Parameter Footprint (FP16):** Each parameter takes up 16 bits (2 bytes) in standard half-precision floating-point format.

$$\text{VRAM} = 70\text{ billion parameters} \times 2\text{ bytes} = 140\text{ GB VRAM}$$

*This requires a cluster of multiple enterprise GPUs, like 2x NVIDIA A100 80GB).*

**Quantization (INT4 / GGUF):** Quantization compresses weights by mapping continuous high-precision floats to lower-bit discrete integers (e.g., down to 4 bits or $0.5\text{ bytes per parameter}$).

$$\text{VRAM (4-bit)} \approx 70\text{ billion} \times 0.5\text{ bytes} \approx 35\text{ GB VRAM}$$

*(Now runnable on a single high-end consumer GPU like an RTX 3090/4090 24GB + system RAM overflow, or an Apple Silicon Mac with 64GB unified memory).*

## 2. Production Implementation Code
Because local runners like Ollama expose an OpenAI-compatible REST API, you do not need a custom SDK. You simply swap the base_url.

In [ ]:
import os
from openai import OpenAI

def query_local_ollama():
    # Point the standard OpenAI client to the local Ollama daemon
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama" # Ollama doesn't require a real key, but the client expects a string
    )
    
    try:
        response = client.chat.completions.create(
            model="llama3", # Assuming llama3 is pulled locally via `ollama pull llama3`
            messages=[
                {"role": "system", "content": "You are a secure, offline corporate assistant."},
                {"role": "user", "content": "State the primary advantage of local LLM deployment in one sentence."}
            ],
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Connection failed. Ensure Ollama is running locally. Error: {e}"

if __name__ == "__main__":
    print("--- Local Model Output (Ollama) ---")
    print(query_local_ollama())

## 3. Deep-Dive: Architecture & Engineering Trade-offs
**Ollama vs. vLLM:** Explain that Ollama is optimized for local developer setup, ease of use, and cross-platform desktop execution (Mac/Windows/Linux). In contrast, high-throughput production backends rely on vLLM, which implements PagedAttention to eliminate memory fragmentation in the KV Cache, dramatically boosting concurrent request throughput.

**Perplexity vs. Speed Degradation:** Lower quantization levels (like 3-bit or 2-bit) drastically drop memory usage but introduce quantization error, degrading model intelligence (perplexity). You must evaluate whether a 4-bit model meets task accuracy benchmarks compared to its unquantized base model.

**Cold Start & Latency:** Loading a 140GB model weights file from disk into VRAM takes significant time. In auto-scaling cloud architectures, local models require persistent instances rather than serverless cold-starts to prevent massive initial timeouts.